# 01 – Data exploration: HASH STI Question & Answer dataset

**Dataset:** *The HASH Sexually Transmitted Infection Question and Answer Crowdsourced Dataset* (2026),
Dataset Club, HASH (Infectious Diseases Institute, Makerere AI Lab and Sunbird AI).
DOI [10.7910/DVN/5SWM5U](https://doi.org/10.7910/DVN/5SWM5U). License **CC BY-NC 4.0**.

**What the dataset page says:** 5,488 Q&A pairs; 8 categories; each pair organised by *intent*
and mapped to two responses (simple and detailed).

**Goal of this notebook:** check what is really in the file before building anything:
1. File structure (sheets, columns, how intents are laid out)
2. Counts: questions, intents, questions per intent, categories
3. How answers are attached to intents
4. Missing values, answer lengths
5. Exact and near-duplicate questions
6. Personal/demographic columns
7. What this means for the train/val/test split

Run `python -m src.download_data` first. This notebook only reads the data.

In [ ]:
import os
import random
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(PROJECT_ROOT)          # so relative paths like data/raw/... work
sys.path.insert(0, PROJECT_ROOT)
from src.load_raw import load_raw_sheets, questions_long

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FIG_DIR = "results/figures"
os.makedirs(FIG_DIR, exist_ok=True)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

## 1. File structure
The Excel file has one sheet per category. We print each sheet's original headers and size.

In [ ]:
sheets = pd.read_excel("data/raw/hash_sti_qa.xlsx", sheet_name=None)
for name, df in sheets.items():
    print(f"{name:<15} rows={len(df):>4}  columns={list(df.columns)}")

In [ ]:
raw = load_raw_sheets()       # all sheets stacked, columns renamed, intent forward-filled
print("Shape:", raw.shape)
raw.dtypes

### How an intent block looks
The intent label is written only on the first row of a block. Questions continue on the rows below.
Below is one block exactly as it appears in Excel (cells shortened to 70 characters).

In [ ]:
def show_block(category, intent):
    block = raw[(raw["category"] == category) & (raw["intent_block"] == intent)]
    cols = ["excel_row", "intent", "unnamed", "q_crowd", "q_other", "simple_answer", "long_answer"]
    return block[cols].apply(lambda col: col.astype(str).str[:70].where(col.notna(), ""))

show_block("Syphilis", "Transmission/Spread_syphilis")

## 2. Basic counts

In [ ]:
q = questions_long(raw)   # one row per question cell

summary = pd.DataFrame({
    "excel_rows": raw.groupby("category").size(),
    "empty_rows": raw.drop(columns=["category", "excel_row", "intent_block"]).isna().all(axis=1).groupby(raw["category"]).sum(),
    "intent_labels": raw.groupby("category")["intent"].count(),
    "unique_intent_labels": raw.groupby("category")["intent"].nunique(),
    "questions": q.groupby("category").size(),
    "q_crowdsourced_col": q[q.question_source == "crowdsourced"].groupby("category").size(),
    "q_another_way_col": q[q.question_source == "another_way"].groupby("category").size(),
    "simple_answers": raw.groupby("category")["simple_answer"].count(),
    "long_answers": raw.groupby("category")["long_answer"].count(),
}).fillna(0).astype(int)
summary.loc["TOTAL"] = summary.sum()
summary

In [ ]:
print("Total question cells:", len(q))
print("Dataset page says   : 5,488 Q&A pairs")

### Cells that contain several questions (line breaks)

In [ ]:
multi = q[q["question"].astype(str).str.contains("\n")]
print("Question cells with line breaks:", len(multi))
print("Lines inside those cells       :", multi["question"].str.split("\n").str.len().sum())
multi.assign(question=multi["question"].str[:200])

## 3. Intent labels: quality checks

In [ ]:
labels = raw.loc[raw["intent"].notna(), ["category", "excel_row", "intent"]].copy()
labels["stripped"] = labels["intent"].str.strip()

print("Labels with extra spaces:", (labels["intent"] != labels["stripped"]).sum())
print(labels[labels["intent"] != labels["stripped"]].to_string(), end="\n\n")

dup_in_sheet = labels[labels.duplicated(["category", "stripped"], keep=False)]
print("Labels repeated inside the same sheet:")
print(dup_in_sheet.to_string(), end="\n\n")

short = labels[labels["stripped"].str.len() <= 10]
print("Very short / generic labels:")
print(short.to_string())

In [ ]:
# All intent labels, per category
labels.groupby("category")["stripped"].apply(list).to_frame("intents")

## 4. How many answers does each intent block have?
The dataset page suggests **one** simple and **one** detailed answer per intent.
We count the answers inside each block.

In [ ]:
blocks = raw.groupby(["category", "intent_block"], sort=False).agg(
    questions=("q_crowd", lambda s: 0),   # placeholder, filled below
    simple_answers=("simple_answer", "count"),
    long_answers=("long_answer", "count"),
)
blocks["questions"] = q.groupby(["category", "intent_block"]).size()
blocks["questions"] = blocks["questions"].fillna(0).astype(int)

one_each = (blocks["simple_answers"] == 1) & (blocks["long_answers"] == 1)
no_answer = (blocks["simple_answers"] == 0) & (blocks["long_answers"] == 0)
print(f"Intent blocks                       : {len(blocks)}")
print(f"Blocks with exactly 1 simple + 1 long: {one_each.sum()}  (holding {blocks.loc[one_each, 'questions'].sum()} questions)")
print(f"Blocks with several answers          : {(~one_each & ~no_answer).sum()}  (holding {blocks.loc[~one_each & ~no_answer, 'questions'].sum()} questions)")
print(f"Blocks with no answer at all         : {no_answer.sum()}")
blocks.sort_values("simple_answers", ascending=False).head(20)

In [ ]:
# Within a multi-answer block, an answer sits on the same row as one question.
# How many questions have an answer on their own row?
has_answer_on_row = raw["simple_answer"].notna() | raw["long_answer"].notna()
q_rows = raw[raw["q_crowd"].notna() | raw["q_other"].notna()]
print("Rows with a question                 :", len(q_rows))
print("...of which have an answer on the row:", has_answer_on_row[q_rows.index].sum())
print("Answer rows with NO question on them :", (has_answer_on_row & raw["q_crowd"].isna() & raw["q_other"].isna()).sum())

In [ ]:
# The 'Unnamed: 1' column: where is it used?
raw.loc[raw["unnamed"].notna(), ["category", "excel_row", "intent_block", "unnamed", "q_other", "simple_answer"]]

## 5. Questions per intent block

In [ ]:
print(blocks["questions"].describe().round(1).to_string(), end="\n\n")
print("Per category (min / median / max questions per intent block):")
blocks.groupby("category")["questions"].agg(["count", "min", "median", "max"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.drop("TOTAL")["questions"].sort_values().plot.barh(ax=axes[0], color="#4C72B0")
axes[0].set_title("Questions per category")
axes[0].set_xlabel("questions")
blocks["questions"].plot.hist(ax=axes[1], bins=30, color="#55A868")
axes[1].set_title("Questions per intent block")
axes[1].set_xlabel("questions in block")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/questions_per_category_and_intent.png", dpi=150)
plt.show()

## 6. Answer lengths (words): simple vs long

In [ ]:
lengths = pd.DataFrame({
    "simple_answer": raw["simple_answer"].dropna().astype(str).str.split().str.len().reset_index(drop=True),
    "long_answer": raw["long_answer"].dropna().astype(str).str.split().str.len().reset_index(drop=True),
})
q_len = q["question"].astype(str).str.split().str.len()
print("Question length (words):")
print(q_len.describe().round(1).to_string(), end="\n\n")
lengths.describe().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist([lengths["simple_answer"].dropna(), lengths["long_answer"].dropna()], bins=30,
        label=["simple", "long"], color=["#4C72B0", "#C44E52"])
ax.set_xlabel("words"); ax.set_ylabel("answers"); ax.legend(); ax.set_title("Answer length")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/answer_lengths.png", dpi=150)
plt.show()

## 7. Exact duplicate questions
Compared after lower-casing, removing punctuation and extra spaces.

In [ ]:
def normalise(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # drop punctuation
    return " ".join(text.split())               # collapse spaces

q["norm"] = q["question"].map(normalise)
dups = q[q.duplicated("norm", keep=False)]
groups = dups.groupby("norm").agg(
    copies=("question", "size"),
    intents=("intent_block", lambda s: sorted(set(s))),
    categories=("category", lambda s: sorted(set(s))),
)
groups["same_intent"] = groups["intents"].str.len() == 1
print("Questions that appear more than once:", groups.shape[0], "distinct texts,", len(dups), "cells")
print("  ...always under the same intent    :", groups["same_intent"].sum())
print("  ...under DIFFERENT intents         :", (~groups["same_intent"]).sum())
groups[~groups["same_intent"]].head(30)

## 8. Near-duplicate questions
TF-IDF on character 3–5-grams, then cosine similarity between every pair of questions.
Pairs ≥ 0.9 are near-identical (typos, punctuation, one word changed).

In [ ]:
unique_q = q.drop_duplicates("norm").reset_index(drop=True)
vectors = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform(unique_q["norm"])
sim = cosine_similarity(vectors)
np.fill_diagonal(sim, 0)

i, j = np.where(np.triu(sim) >= 0.9)
near = pd.DataFrame({
    "question_a": unique_q.loc[i, "question"].values,
    "question_b": unique_q.loc[j, "question"].values,
    "similarity": sim[i, j].round(3),
    "intent_a": unique_q.loc[i, "intent_block"].values,
    "intent_b": unique_q.loc[j, "intent_block"].values,
})
near["same_intent"] = near["intent_a"] == near["intent_b"]
print("Near-duplicate pairs (>= 0.9):", len(near), "| same intent:", near["same_intent"].sum())
near.sort_values("similarity", ascending=False).head(15)

In [ ]:
# Near-duplicates under DIFFERENT intents: these are label conflicts
near[~near["same_intent"]].head(20)

## 9. Personal / demographic columns

In [ ]:
all_headers = sorted({h for df in sheets.values() for h in df.columns})
print("All column headers in the file:")
for h in all_headers:
    print(" -", h)

## 10. Summary of findings (fill in after reading the outputs)
| Check | Result |
|---|---|
| Question cells (vs 5,488 claimed) | |
| Intent blocks | |
| Blocks with exactly 1 simple + 1 long answer | |
| Personal/demographic columns | |
| Duplicate questions under different intents | |

**Decision needed before Phase 2:** what is the label (the "intent") the model predicts,
given that many blocks hold several different answers?